# Visualize Reconstruction Slices for Paper Figure

This notebook creates orthogonal slice visualizations (XY, XZ, YZ) comparing MissAlignment results with the AreTomo2 baseline.

In [49]:
# Configuration
MODEL_ID = 2  # SHREC model ID (for output filenames)

# XML metadata paths
MISS_ALIGNMENT_XML = "/home/marten/data/datasets/shrec_benchmark/iter6/model_2.xml"
BASELINE_XML = "/home/marten/data/datasets/shrec_benchmark/at2_aln/model_2.xml"

# Reconstruction settings
OVERSAMPLING = 4.0

# Output settings
DPI = 300

# Darkest voxel search settings
EDGE_MARGIN = 64  # Minimum distance from edges for finding darkest voxel

In [50]:
import numpy as np
import torch
from pathlib import Path
import matplotlib.pyplot as plt

from miss_alignment.data.io import TiltSeriesData

In [51]:
# Load tilt series data from XML files
miss_alignment_data = TiltSeriesData(xml_metadata_path=Path(MISS_ALIGNMENT_XML))
baseline_data = TiltSeriesData(xml_metadata_path=Path(BASELINE_XML))

print(f"Loading MissAlignment: {MISS_ALIGNMENT_XML}")
ts_miss, images_miss, pixel_size = miss_alignment_data.load_metadata_and_stack()

print(f"Loading Baseline: {BASELINE_XML}")
ts_baseline, images_baseline, _ = baseline_data.load_metadata_and_stack()

print(f"\nImages shape: {images_miss.shape}")
print(f"Pixel size: {pixel_size:.2f} A")

Loading MissAlignment: /home/marten/data/datasets/shrec_benchmark/iter6/model_2.xml
Loading Baseline: /home/marten/data/datasets/shrec_benchmark/at2_aln/model_2.xml

Images shape: torch.Size([61, 512, 512])
Pixel size: 10.00 A


In [52]:
# Get volume dimensions from metadata
volume_dims = ts_miss.volume_dimensions_physical  # (X, Y, Z) in Angstroms
center_angstrom = volume_dims / 2
print(f"Volume dimensions: {volume_dims.tolist()} A")
print(f"Volume center: {center_angstrom.tolist()} A")

Volume dimensions: [5120.0, 5120.0, 1800.0] A
Volume center: [2560.0, 2560.0, 900.0] A


In [54]:
# Calculate full volume size from metadata
volume_dims_voxels = (ts_miss.volume_dimensions_physical / pixel_size).int()
print(f"Full volume size (voxels): {volume_dims_voxels.tolist()}")

# Reconstruct full volumes using warpylib's tiled weighted backprojection
# Note: reconstruct_full handles normalization internally
device = "cpu"

# Move data to device
images_miss_device = images_miss.to(device)
images_baseline_device = images_baseline.to(device)
ts_miss_device = ts_miss.to(device)
ts_baseline_device = ts_baseline.to(device)

# Get volume dimensions as tuple (required by reconstruct_full)
volume_dimensions_physical = tuple(ts_miss.volume_dimensions_physical.tolist())

print(f"\nReconstructing MissAlignment volume (oversampling={OVERSAMPLING})...")
vol_miss_alignment = ts_miss_device.reconstruct_full(
    tilt_data=images_miss_device,
    pixel_size=pixel_size,
    volume_dimensions_physical=volume_dimensions_physical,
    subvolume_size=64,
    subvolume_oversampling=OVERSAMPLING,
    normalize=True,
    invert=False,
    apply_ctf=False,
    ctf_weighted=False,
    correct_attenuation=True,
    batch_size=8,
).cpu().numpy()

print(f"Reconstructing Baseline volume (oversampling={OVERSAMPLING})...")
vol_baseline = ts_baseline_device.reconstruct_full(
    tilt_data=images_baseline_device,
    pixel_size=pixel_size,
    volume_dimensions_physical=volume_dimensions_physical,
    subvolume_size=64,
    subvolume_oversampling=OVERSAMPLING,
    normalize=True,
    invert=False,
    apply_ctf=False,
    ctf_weighted=False,
    correct_attenuation=True,
    batch_size=8,
).cpu().numpy()

print(f"\nMissAlignment volume shape: {vol_miss_alignment.shape}")
print(f"Baseline volume shape: {vol_baseline.shape}")

Full volume size (voxels): [512, 512, 180]

Reconstructing MissAlignment volume (oversampling=4.0)...
Reconstructing Baseline volume (oversampling=4.0)...

MissAlignment volume shape: (180, 512, 512)
Baseline volume shape: (180, 512, 512)


In [59]:
# Find the darkest voxel in the MissAlignment volume (avoiding edges)
z_dim, y_dim, x_dim = vol_miss_alignment.shape

# Create a mask for valid region (at least EDGE_MARGIN from edges)
valid_volume = vol_miss_alignment[
    EDGE_MARGIN : z_dim - EDGE_MARGIN,
    EDGE_MARGIN : y_dim - EDGE_MARGIN,
    EDGE_MARGIN : x_dim - EDGE_MARGIN
]

# Find the minimum (darkest) voxel in the valid region
min_idx_flat = np.argmin(valid_volume)
min_idx_local = np.unravel_index(min_idx_flat, valid_volume.shape)

# Convert back to global coordinates
darkest_z = min_idx_local[0] + EDGE_MARGIN
darkest_y = min_idx_local[1] + EDGE_MARGIN
darkest_x = min_idx_local[2] + EDGE_MARGIN

print(f"Volume shape: {vol_miss_alignment.shape}")
print(f"Darkest voxel location: z={darkest_z}, y={darkest_y}, x={darkest_x}")
print(f"Darkest voxel value: {vol_miss_alignment[darkest_z, darkest_y, darkest_x]:.4f}")

Volume shape: (180, 512, 512)
Darkest voxel location: z=80, y=319, x=306
Darkest voxel value: -36.0289


In [63]:
def extract_slices_at_point(volume, z, y, x, n_slices=6):
    """Extract XY, XZ, and YZ slices at a specific point in the volume.
    
    Averages n_slices adjacent slices centered on the specified coordinate
    to improve contrast.
    
    Volume is assumed to be in ZYX order.
    
    Returns:
        tuple: (xy_slice, xz_slice, yz_slice)
    """
    half = n_slices // 2
    
    # XY slice: average slices along Z axis
    z_start = max(0, z - half)
    z_end = min(volume.shape[0], z_start + n_slices)
    xy_slice = volume[z_start:z_end, :, :].mean(axis=0)  # shape: (Y, X)
    
    # XZ slice: average slices along Y axis
    y_start = max(0, y - half)
    y_end = min(volume.shape[1], y_start + n_slices)
    xz_slice = volume[:, y_start:y_end, :].mean(axis=1)  # shape: (Z, X)
    
    # YZ slice: average slices along X axis
    x_start = max(0, x - half)
    x_end = min(volume.shape[2], x_start + n_slices)
    yz_slice = volume[:, :, x_start:x_end].mean(axis=2)  # shape: (Z, Y)
    
    return xy_slice, xz_slice, yz_slice

# Extract slices at the darkest voxel location for both volumes (averaging 10 slices)
miss_alignment_slices = extract_slices_at_point(vol_miss_alignment, darkest_z, darkest_y, darkest_x, n_slices=10)
baseline_slices = extract_slices_at_point(vol_baseline, darkest_z, darkest_y, darkest_x, n_slices=10)

print("Extracted mean projection of 10 adjacent slices at darkest voxel location:")
print(f"  XY slice shape: {miss_alignment_slices[0].shape}")
print(f"  XZ slice shape: {miss_alignment_slices[1].shape}")
print(f"  YZ slice shape: {miss_alignment_slices[2].shape}")

Extracted mean projection of 10 adjacent slices at darkest voxel location:
  XY slice shape: (512, 512)
  XZ slice shape: (180, 512)
  YZ slice shape: (180, 512)


In [64]:
def normalize_by_std(image, n_std=3):
    """Normalize image to [-n_std, +n_std] standard deviations, then scale to [0, 1].
    
    Args:
        image: 2D numpy array
        n_std: Number of standard deviations for clipping range
    
    Returns:
        Normalized image in range [0, 1]
    """
    mean = np.mean(image)
    std = np.std(image)
    
    # Clip to [-n_std, +n_std] standard deviations
    vmin = mean - n_std * std
    vmax = mean + n_std * std
    
    # Normalize to [0, 1]
    normalized = np.clip((image - vmin) / (vmax - vmin), 0, 1)
    
    return normalized

# Normalize all slices
miss_alignment_normalized = tuple(normalize_by_std(s) for s in miss_alignment_slices)
baseline_normalized = tuple(normalize_by_std(s) for s in baseline_slices)

print("Normalized slices to [-5, +5] std range")

Normalized slices to [-5, +5] std range


In [65]:
# Save each slice as a separate SVG
slice_names = ['xy', 'xz', 'yz']

def save_slice_as_svg(image, filename):
    """Save a single slice as SVG with no decorations."""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='spline36')
    ax.axis('off')
    
    fig.savefig(
        filename,
        format='svg',
        dpi=DPI,
        transparent=True,
        bbox_inches='tight',
        pad_inches=0
    )
    plt.close(fig)
    print(f"Saved: {filename}")

# Save baseline slices
for name, image in zip(slice_names, baseline_normalized):
    filename = f"model_{MODEL_ID}_baseline_{name}.svg"
    save_slice_as_svg(image, filename)

# Save MissAlignment slices
for name, image in zip(slice_names, miss_alignment_normalized):
    filename = f"model_{MODEL_ID}_missalignment_{name}.svg"
    save_slice_as_svg(image, filename)

Saved: model_2_baseline_xy.svg
Saved: model_2_baseline_xz.svg
Saved: model_2_baseline_yz.svg
Saved: model_2_missalignment_xy.svg
Saved: model_2_missalignment_xz.svg
Saved: model_2_missalignment_yz.svg


In [87]:

def rasterize_volume(vol, band_width=30, spacing=160, value=0):
    """Set bands of pixels to 0 along each dimension for visualization."""
    result = vol.copy()
    for axis in range(3):
        for start in range(0, vol.shape[axis], spacing):
            slices = [slice(None)] * 3
            slices[axis] = slice(start, min(start + band_width, vol.shape[axis]))
            result[tuple(slices)] = value
    
    times = np.array(vol.shape) // spacing
    result = result[:times[0] * spacing, :times[1] * spacing, :times[2] * spacing]
    return result

rast = rasterize_volume(vol_miss_alignment, value=vol_miss_alignment.max())

In [88]:
import napari
viewer = napari.Viewer()
viewer.add_image(rast, contrast_limits=[-3, 3])

<Image layer 'rast' at 0x742ad02aba50>